In [ ]:
pip install ultralytics

In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


dataset_path = "/kaggle/input/license-plate-dataset"
train_images = os.listdir(f"{dataset_path}/images/train")
val_images = os.listdir(f"{dataset_path}/images/val")
train_labels = os.listdir(f"{dataset_path}/labels/train")
val_labels = os.listdir(f"{dataset_path}/labels/val")

print(f"Number of train images: {len(train_images)}")
print(f"Number of val images: {len(val_images)}")
print(f"Number of train labels: {len(train_labels)}")
print(f"Number of val labels: {len(val_labels)}")
print("Sample train image-label pairs:", list(zip(train_images[:5], train_labels[:5])))
print("Sample val image-label pairs:", list(zip(val_images[:5], val_labels[:5])))

In [ ]:
import os
import shutil
import random

# Define paths
input_dataset_path = "/kaggle/input/license-plate-dataset"
output_dataset_path = "/kaggle/working/reduced_dataset"
os.makedirs(f"{output_dataset_path}/images/train", exist_ok=True)
os.makedirs(f"{output_dataset_path}/images/val", exist_ok=True)
os.makedirs(f"{output_dataset_path}/labels/train", exist_ok=True)
os.makedirs(f"{output_dataset_path}/labels/val", exist_ok=True)

# Get lists of image and label files
train_images = sorted(os.listdir(f"{input_dataset_path}/images/train"))
val_images = sorted(os.listdir(f"{input_dataset_path}/images/val"))
train_labels = sorted(os.listdir(f"{input_dataset_path}/labels/train"))
val_labels = sorted(os.listdir(f"{input_dataset_path}/labels/val"))

# Ensure the order matches
train_pairs = list(zip(train_images, train_labels))
val_pairs = list(zip(val_images, val_labels))

# Randomly select 300 train pairs and 100 val pairs
random.seed(42)
train_pairs_subset = random.sample(train_pairs, 300)
val_pairs_subset = random.sample(val_pairs, 100)

# Copy the selected train files
for img_file, lbl_file in train_pairs_subset:
    shutil.copy(f"{input_dataset_path}/images/train/{img_file}", f"{output_dataset_path}/images/train/{img_file}")
    shutil.copy(f"{input_dataset_path}/labels/train/{lbl_file}", f"{output_dataset_path}/labels/train/{lbl_file}")

# Copy the selected val files
for img_file, lbl_file in val_pairs_subset:
    shutil.copy(f"{input_dataset_path}/images/val/{img_file}", f"{output_dataset_path}/images/val/{img_file}")
    shutil.copy(f"{input_dataset_path}/labels/val/{lbl_file}", f"{output_dataset_path}/labels/val/{lbl_file}")

# Verify the new counts
print(f"Reduced train images: {len(os.listdir(f'{output_dataset_path}/images/train'))}")
print(f"Reduced val images: {len(os.listdir(f'{output_dataset_path}/images/val'))}")
print(f"Reduced train labels: {len(os.listdir(f'{output_dataset_path}/labels/train'))}")
print(f"Reduced val labels: {len(os.listdir(f'{output_dataset_path}/labels/val'))}")

In [ ]:
from ultralytics import YOLO
import os

# Define the path to the reduced dataset
reduced_dataset_path = "/kaggle/working/reduced_dataset"

# Create data.yaml file
data_yaml_content = f"""
train: {reduced_dataset_path}/images/train
val: {reduced_dataset_path}/images/val
test: {reduced_dataset_path}/images/val

nc: 1
names: ['number_plate']
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml_content)

# Load the model
model = YOLO("yolov8n.pt")  # Nano model for speed

# Train the model with optimized settings
model.train(
    data="data.yaml",
    epochs=10,  # Very short training
    imgsz=320,  # Small image size
    batch=4,  # Small batch size
    project="runs/train",
    name="number_plate_detection"
)

# Validate the model
model.val(data="data.yaml")

# Predict on a test image
val_images = os.listdir(f"{reduced_dataset_path}/images/val")
print("Sample images in val folder:", val_images[:5])
test_image = f"{reduced_dataset_path}/images/val/{val_images[0]}" if val_images else None
if test_image:
    results = model.predict(source=test_image)
    results[0].save()
else:
    print("No images found in val folder for prediction.")